# 🚀 Kaggle 00: Self-Extract CLIP Feature Vectors

Trích xuất vector embedding từ ảnh keyframe bằng mô hình CLIP mạnh nhất hiện nay.

**Mô hình được chọn:** `EVA02-E-14-plus` (OpenCLIP) — top benchmark trên COCO & Flickr30k  
**Output dim:** 1024-dim  
**Output:** Các file `.npy` per-video lưu vào `/kaggle/working/clip_features/`

In [ ]:
# ── Cell 1: Cài thư viện ──────────────────────────────────────
!pip install -q open-clip-torch timm

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────
import os

# ⚠️ Thay đường dẫn này cho đúng với dataset của bạn trên Kaggle
KEYFRAMES_ROOT = "/kaggle/input/aic-hcmc-data/keyframes/keyframes"
MAP_KEYFRAMES_DIR = "/kaggle/input/aic-hcmc-data/map-keyframes-aic25-b1/map-keyframes"
OUTPUT_DIR = "/kaggle/working/clip_features"

# Mô hình CLIP — chọn 1 trong 3 tùy VRAM:
# - "EVA02-E-14-plus" / "laion2b_s9b_b144k"  → MẠNH NHẤT, dim=1024, cần ~18GB VRAM
# - "ViT-L-14"        / "datacomp_xl_s13b_b90k" → TRUNG BÌNH, dim=768, cần ~8GB VRAM
# - "ViT-B-32"        / "openai"              → NHẸ NHẤT, dim=512, cần ~3GB VRAM
MODEL_NAME  = "EVA02-E-14-plus"
PRETRAINED  = "laion2b_s9b_b144k"
BATCH_SIZE  = 64    # Giảm xuống 32 nếu bị OOM
NUM_WORKERS = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Model: {MODEL_NAME} | Pretrained: {PRETRAINED}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# ── Cell 3: Load Model ────────────────────────────────────────
import open_clip
import torch
import numpy as np
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED, device=DEVICE
)
model.eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

# Lấy số chiều output
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224).to(DEVICE)
    feat_dim = model.encode_image(dummy).shape[-1]

print(f"✅ Model loaded — Feature dim: {feat_dim}")

In [ ]:
# ── Cell 4: Quick Sanity Test ─────────────────────────────────
# Kiểm tra text-image alignment trước khi chạy full
import glob

# Lấy ảnh bất kỳ để test
test_imgs = glob.glob(f"{KEYFRAMES_ROOT}/**/*.jpg", recursive=True)[:1]
assert test_imgs, "Không tìm thấy ảnh keyframe! Kiểm tra lại KEYFRAMES_ROOT."

test_queries = [
    "a person speaking at a podium",
    "a soccer match on green grass",
    "a news broadcast on TV",
]

img_tensor = preprocess(Image.open(test_imgs[0]).convert("RGB")).unsqueeze(0).to(DEVICE)
text_tokens = tokenizer(test_queries).to(DEVICE)

with torch.no_grad():
    img_feat  = model.encode_image(img_tensor)
    img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
    text_feat = model.encode_text(text_tokens)
    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
    sims = (img_feat @ text_feat.T).squeeze().cpu().numpy()

print(f"\n✅ Sanity test image: {test_imgs[0]}")
for q, s in zip(test_queries, sims):
    print(f"   Cosine({q!r}): {s:.4f}")

In [ ]:
# ── Cell 5: Trích xuất Feature Vectors toàn bộ keyframes ──────
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class KeyframeDataset(Dataset):
    """Dataset đọc các ảnh .jpg từ 1 video dựa trên file CSV."""
    def __init__(self, img_paths, preprocess):
        self.paths = img_paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert("RGB")
            return self.preprocess(img), str(path)
        except Exception:
            # Trả về ảnh đen nếu bị lỗi, tránh crash cả batch
            return torch.zeros(3, 224, 224), str(path)


def extract_one_video(csv_path: Path, video_id: str, out_dir: str) -> int:
    """Trích xuất và lưu file .npy cho 1 video. Trả về số frame đã xử lý."""
    out_npy = Path(out_dir) / f"{video_id}.npy"
    if out_npy.exists():
        return 0  # Đã có, bỏ qua (resume)

    df = pd.read_csv(csv_path)
    batch_id = video_id.split("_")[0]

    # Dựng danh sách đường dẫn ảnh theo thứ tự n tăng dần (QUAN TRỌNG!)
    img_paths = []
    for n in sorted(df["n"].tolist()):
        base = Path(KEYFRAMES_ROOT) / f"Keyframes_{batch_id}" / "keyframes" / video_id
        found = None
        for cand in [f"{n:03d}.jpg", f"{n}.jpg", f"{n:04d}.jpg"]:
            p = base / cand
            if p.exists():
                found = str(p)
                break
        img_paths.append(found or "")

    # Loại bỏ frame không tìm thấy ảnh
    valid_paths = [p for p in img_paths if p]
    if len(valid_paths) != len(img_paths):
        missing = len(img_paths) - len(valid_paths)
        print(f"  ⚠️  {video_id}: thiếu {missing} ảnh — bỏ qua các frame này")

    if not valid_paths:
        return 0

    ds = KeyframeDataset(valid_paths, preprocess)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                        pin_memory=True, shuffle=False)

    all_vecs = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(DEVICE)
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_vecs.append(feats.cpu().float().numpy())

    vectors = np.vstack(all_vecs)  # shape: (num_frames, feat_dim)
    np.save(str(out_npy), vectors)
    return len(vectors)


# ── Chạy toàn bộ các video ────────────────────────────────────
csv_files = sorted(Path(MAP_KEYFRAMES_DIR).glob("*.csv"))
print(f"Found {len(csv_files)} videos to process")

total_frames = 0
errors = []

for csv_path in tqdm(csv_files, desc="Extracting videos"):
    video_id = csv_path.stem
    try:
        n = extract_one_video(csv_path, video_id, OUTPUT_DIR)
        total_frames += n
    except Exception as e:
        errors.append((video_id, str(e)))
        print(f"  ❌ Error on {video_id}: {e}")

print(f"\n✅ Done! Total frames extracted: {total_frames:,}")
print(f"   Errors: {len(errors)} videos")
if errors:
    for vid, err in errors[:10]:
        print(f"   {vid}: {err}")

In [ ]:
# ── Cell 6: Xác minh kết quả (Verification) ──────────────────
npy_files = sorted(Path(OUTPUT_DIR).glob("*.npy"))
print(f"Số file .npy đã tạo: {len(npy_files)}")
print(f"Số file .csv gốc   : {len(csv_files)}")

# So sánh tên file
npy_names = {f.stem for f in npy_files}
csv_names = {f.stem for f in csv_files}
missing_npy = csv_names - npy_names
if missing_npy:
    print(f"⚠️  Thiếu {len(missing_npy)} file .npy: {list(missing_npy)[:5]}")
else:
    print("✅ Tên file .npy khớp 100% với .csv")

# Kiểm tra số vector trong .npy vs số dòng trong .csv
print("\nKiểm tra per-video frame count:")
mismatch = 0
for csv_path in list(csv_files)[:10]:  # Thử 10 video đầu
    vid = csv_path.stem
    npy_path = Path(OUTPUT_DIR) / f"{vid}.npy"
    if not npy_path.exists():
        continue
    csv_rows = len(pd.read_csv(csv_path))
    npy_rows = np.load(str(npy_path)).shape[0]
    match = "✅" if csv_rows == npy_rows else "❌ MISMATCH"
    if csv_rows != npy_rows:
        mismatch += 1
    print(f"  {match}  {vid}: CSV={csv_rows} rows | NPY={npy_rows} vectors")

if mismatch == 0:
    print("\n✅ Tất cả vector count khớp với CSV. Dữ liệu CHUẨN!")
else:
    print(f"\n⚠️  Có {mismatch} video bị mismatch — cần kiểm tra lại!")

# In info một vector mẫu
sample = np.load(str(npy_files[0]))
print(f"\nVector sample ({npy_files[0].stem}):")
print(f"  Shape : {sample.shape}")
print(f"  Dtype : {sample.dtype}")
print(f"  Norm[0]: {np.linalg.norm(sample[0]):.5f}  (phải ≈ 1.0 nếu đã L2-normalized)")

In [ ]:
# ── Cell 7: Dựng FAISS Index mới từ vector vừa trích xuất ─────
import sys
sys.path.insert(0, "/kaggle/input/aic-system-code/AIC_System")

import faiss

FEAT_DIM    = feat_dim          # lấy từ Cell 3 (512 / 768 / 1024)
INDEX_DIR   = "/kaggle/working/indexes"
os.makedirs(INDEX_DIR, exist_ok=True)

# Build FAISS HNSW index
hnsw = faiss.IndexHNSWFlat(FEAT_DIM, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efSearch = 64
index_new = faiss.IndexIDMap(hnsw)

current_id = 0
for npy_path in tqdm(sorted(Path(OUTPUT_DIR).glob("*.npy")), desc="Building FAISS"):
    vecs = np.load(str(npy_path)).astype(np.float32)
    # Đã L2-normalized rồi, không cần normalize lại
    ids = np.arange(current_id, current_id + len(vecs), dtype=np.int64)
    index_new.add_with_ids(vecs, ids)
    current_id += len(vecs)

faiss_out = f"{INDEX_DIR}/faiss_visual.index"
faiss.write_index(index_new, faiss_out)
print(f"\n✅ FAISS Index saved: {index_new.ntotal:,} vectors → {faiss_out}")

In [ ]:
# ── Cell 8: Dựng keyframe_master.parquet ─────────────────────
# Build metadata parquet theo đúng thứ tự npy files
records = []
faiss_id = 0

for npy_path in sorted(Path(OUTPUT_DIR).glob("*.npy")):
    video_id = npy_path.stem
    batch_id = video_id.split("_")[0]
    csv_path = Path(MAP_KEYFRAMES_DIR) / f"{video_id}.csv"
    if not csv_path.exists():
        continue

    df_kf = pd.read_csv(csv_path).sort_values("n").reset_index(drop=True)
    npy_count = np.load(str(npy_path)).shape[0]

    # Chỉ lấy đúng số dòng khớp với số vector
    df_kf = df_kf.head(npy_count)

    base_dir = Path(KEYFRAMES_ROOT) / f"Keyframes_{batch_id}" / "keyframes" / video_id
    for _, row in df_kf.iterrows():
        n = int(row["n"])
        img_name = f"{n:03d}.jpg"
        for cand in [f"{n:03d}.jpg", f"{n}.jpg"]:
            if (base_dir / cand).exists():
                img_name = cand
                break
        records.append({
            "faiss_id":    faiss_id,
            "keyframe_id": f"{video_id}_n{n}",
            "video_id":    video_id,
            "batch_id":    batch_id,
            "n":           n,
            "frame_idx":   int(row["frame_idx"]),
            "pts_time":    float(row["pts_time"]),
            "fps":         float(row["fps"]),
            "image_path":  str(base_dir / img_name),
        })
        faiss_id += 1

df_master = pd.DataFrame(records)
parquet_out = f"{INDEX_DIR}/keyframe_master.parquet"
df_master.to_parquet(parquet_out, index=False)

print(f"✅ keyframe_master.parquet saved: {len(df_master):,} rows → {parquet_out}")

# Final sanity check
faiss_count = index_new.ntotal
parquet_count = len(df_master)
status = "✅ KHỚP" if faiss_count == parquet_count else "❌ MISMATCH"
print(f"\n{status} — FAISS: {faiss_count:,} | Parquet: {parquet_count:,}")

In [ ]:
# ── Cell 9: Nén & tải về ─────────────────────────────────────
!cd /kaggle/working && zip -q indexes.zip indexes/faiss_visual.index indexes/keyframe_master.parquet
print("✅ Đã nén xong → /kaggle/working/indexes.zip")

# Xem kích thước
!ls -lh /kaggle/working/indexes.zip
!ls -lh /kaggle/working/indexes/